In [ ]:
import torch
import numpy as np
import yfinance as yf
from collections import deque
import random
import matplotlib.pyplot as plt
from env_trading import SecureTradingEnv
from networks import Actor, Critic, DEVICE
import time
from tqdm import trange
import pandas as pd
start_time = time.time()

# -------- GPU SPEED BOOST --------
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("Device:", DEVICE)

# ---------------- TD3 HYPERPARAMS ----------------
# train.py - Hyperparameter tweaks
policy_noise = 0.2
noise_clip = 0.5
policy_delay = 3 # Increased from 2 to allow critic to be even more stable
gamma = 0.99
tau = 0.005
batch_size = 512
max_explore_steps = 50_000
episodes = 4500   # 🔥 can increase later

# ---------------- DATA ----------------
# SWAPPED HDFCBANK.NS for ICICIBANK.NS because we have news for ICICI
assets = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "ICICIBANK.NS", "ITC.NS"]
full_data = {}

print("Downloading full training data (2010-2023)...")
try:
    print("Loading sentiment_data.csv...")
    sent_df = pd.read_csv("sentiment_data.csv")
    sent_df["date"] = pd.to_datetime(sent_df["date"]).dt.normalize() # Ensure dates are normalized
    sent_df.set_index("date", inplace=True)
    print(f"Sentiment data loaded: {len(sent_df)} rows.")
except FileNotFoundError:
    print("WARNING: sentiment_data.csv NOT FOUND! Training with dummy sentiment (0.0).")
    sent_df = None
for asset in assets:
    df = yf.download(asset, start="2010-01-01", end="2023-01-01",
                 auto_adjust=True, progress=False)
    df.columns = df.columns.get_level_values(0)
    
    # ===== BASIC =====
    df["Returns"] = df["Close"].pct_change()
    
    # ===== RSI =====
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    df["RSI"] = 100 - (100 / (1 + rs))
    
    # ===== MACD =====
    ema12 = df["Close"].ewm(span=12).mean()
    ema26 = df["Close"].ewm(span=26).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9).mean()
    
    # ===== SMA / EMA =====
    df["SMA20"] = df["Close"].rolling(20).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()
    df["EMA50"] = df["Close"].ewm(span=50).mean()
    
    # ===== Bollinger Bands =====
    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    
    df["SMA20"] = sma20
    df["BB_upper"] = sma20 + (2 * std20)
    df["BB_lower"] = sma20 - (2 * std20)
    
    # ===== ATR =====
    high_low = df["High"] - df["Low"]
    high_close = np.abs(df["High"] - df["Close"].shift())
    low_close = np.abs(df["Low"] - df["Close"].shift())
    true_range = np.maximum(high_low, np.maximum(high_close, low_close))
    df["ATR"] = true_range.rolling(14).mean()
    
    # ===== Momentum =====
    df["Momentum"] = df["Close"] - df["Close"].shift(10)
    
    # ===== OBV =====
    obv = [0]
    for i in range(1, len(df)):
        if df["Close"].iloc[i] > df["Close"].iloc[i-1]:
            obv.append(obv[-1] + df["Volume"].iloc[i])
        elif df["Close"].iloc[i] < df["Close"].iloc[i-1]:
            obv.append(obv[-1] - df["Volume"].iloc[i])
        else:
            obv.append(obv[-1])
    df["OBV"] = obv
    
    # ===== Dummy sentiment =====
    if sent_df is not None:
        # Join sentiment data based on date
        df = df.join(sent_df["sentiment_smooth"], how="left")
        # Fill missing days (e.g., weekends/no-news days) with 0.0
        df["Sentiment"] = df["sentiment_smooth"].fillna(0.0)
        # Drop the temp column
        df.drop(columns=["sentiment_smooth"], inplace=True)
    else:
        df["Sentiment"] = 0.0
    
    df.dropna(inplace=True)
    full_data[asset] = df

print("Data ready.")

years = list(range(2010, 2023))

# -------- TEMP ENV to get dimensions --------
sample_year = "2020"
temp_data = {}
for a in assets:
    year_df = full_data[a].loc[str(sample_year)]

    # if empty, choose another year
    if len(year_df) < 200:
        year_df = full_data[a].sample(300)

    temp_data[a] = year_df
temp_env = SecureTradingEnv(temp_data, assets)

state_dim = temp_env.observation_space.shape[0]
action_dim = temp_env.action_space.shape[0]

# ---------------- NETWORKS ----------------
actor = Actor(state_dim, action_dim).to(DEVICE)
critic1 = Critic(state_dim, action_dim).to(DEVICE)
critic2 = Critic(state_dim, action_dim).to(DEVICE)

actor_target = Actor(state_dim, action_dim).to(DEVICE)
critic1_target = Critic(state_dim, action_dim).to(DEVICE)
critic2_target = Critic(state_dim, action_dim).to(DEVICE)

actor_target.load_state_dict(actor.state_dict())
critic1_target.load_state_dict(critic1.state_dict())
critic2_target.load_state_dict(critic2.state_dict())

actor_opt = torch.optim.Adam(actor.parameters(), lr=5e-5)
critic_opt = torch.optim.Adam(
    list(critic1.parameters()) + list(critic2.parameters()), lr=5e-5
)

# ---------------- REPLAY BUFFER ----------------
buffer = deque(maxlen=1_000_000)

def sample_batch():
    batch = random.sample(buffer, batch_size)
    s, a, r, s_, d = zip(*batch)

    return (
        torch.as_tensor(np.array(s), dtype=torch.float32, device=DEVICE),
        torch.as_tensor(np.array(a), dtype=torch.float32, device=DEVICE),
        torch.as_tensor(r, dtype=torch.float32, device=DEVICE).unsqueeze(1),
        torch.as_tensor(np.array(s_), dtype=torch.float32, device=DEVICE),
        torch.as_tensor(d, dtype=torch.float32, device=DEVICE).unsqueeze(1),
    )

# ---------------- TRAINING ----------------
episode_profits = []
episode_rewards = []
total_steps = 0
best_profit = -1e12
for episode in trange(episodes, desc="Training Progress"):

    # 🔥 RANDOM YEAR TRAINING (CRITICAL FOR SPEED + PAPER)
    start_year = np.random.randint(2010, 2021)
    end_year = start_year + 1
    year_range = f"{start_year}-{end_year}"

    episode_data = {}

    for a in assets:
        df_year = full_data[a].loc[str(start_year):str(end_year)]
    
        # if too small, fallback random window
        if len(df_year) < 200:
            df_year = full_data[a].sample(300)
    
        episode_data[a] = df_year

    env = SecureTradingEnv(episode_data, assets)

    state, _ = env.reset()
    ep_reward = 0.0
    start_net_worth = env.net_worth

    while True:
        total_steps += 1

        # -------- ACTION --------
        if total_steps < max_explore_steps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = actor(
                    torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                ).cpu().numpy()[0]

            noise = max(0.25 * (1 - episode/2000), 0.05)
            action += np.random.normal(0, noise, size=action.shape)
            action = np.clip(action, -1, 1)

        next_state, reward, done, _, _ = env.step(action)

        buffer.append((state, action, reward, next_state, done))

        state = next_state
        ep_reward += reward

        # -------- LEARNING --------
        if len(buffer) >= batch_size:
            s, a, r, s_, d = sample_batch()

            with torch.no_grad():
                noise = (torch.randn_like(a) * policy_noise).clamp(-noise_clip, noise_clip)
                a_next = (actor_target(s_) + noise).clamp(-1, 1)

                q1_next = critic1_target(s_, a_next)
                q2_next = critic2_target(s_, a_next)
                q_target = r + gamma * (1 - d) * torch.min(q1_next, q2_next)

            q1 = critic1(s, a)
            q2 = critic2(s, a)
            critic_loss = ((q1 - q_target) ** 2 + (q2 - q_target) ** 2).mean()

            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()

            if total_steps % policy_delay == 0:
                actor_loss = -critic1(s, actor(s)).mean()

                actor_opt.zero_grad()
                actor_loss.backward()
                actor_opt.step()

                # target updates
                for p, tp in zip(actor.parameters(), actor_target.parameters()):
                    tp.data.copy_(tau * p.data + (1 - tau) * tp.data)

                for net, net_t in zip([critic1, critic2],[critic1_target, critic2_target]):
                    for p, tp in zip(net.parameters(), net_t.parameters()):
                        tp.data.copy_(tau * p.data + (1 - tau) * tp.data)

        if done:
            break

    episode_profit = env.net_worth - start_net_worth
    episode_rewards.append(ep_reward)
    episode_profits.append(episode_profit)
    if env.net_worth > best_profit:
        best_profit = env.net_worth
        torch.save(actor.state_dict(), "BEST_MODEL.pth")

    # -------- PRINT EVERY 50 --------
    if episode % 50 == 0 and episode > 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"\nEpisode {episode} | "
            f"AvgProfit(50): {np.mean(episode_profits[-50:]):.2f} | "
            f"AvgReward(50): {np.mean(episode_rewards[-50:]):.2f} | "
            f"Time: {elapsed:.1f} mins"
        )
    # ⭐ SAVE MODEL EVERY 500 EPISODES (ADD THIS)
    if episode % 500 == 0 and episode > 0:
        torch.save(actor.state_dict(), f"actor_ep{episode}.pth")
        print(f"Checkpoint saved at episode {episode}")
print("\n====== FINAL SUMMARY ======")
print("Final 50 avg reward:", np.mean(episode_rewards[-50:]))
print("Final 50 avg profit:", np.mean(episode_profits[-50:]))
print("Best profit:", np.max(episode_profits))
print("Worst profit:", np.min(episode_profits))

# -------- SAVE MODEL --------
torch.save(actor.state_dict(), "td3_latest_trader_actor_final.pth")
torch.save(critic1.state_dict(), "td3_latest_trader_critic1_final.pth")
torch.save(critic2.state_dict(), "td3_latest_trader_critic2_final.pth")

print("\nMODEL SAVED SUCCESSFULLY")

# -------- PLOT --------
plt.figure(figsize=(10,5))
plt.plot(episode_profits)
plt.title("Training Profit Curve")
plt.xlabel("Episode")
plt.ylabel("Profit")
plt.grid()
plt.show()

import pandas as pd

df_metrics = pd.DataFrame({
    "Episode": list(range(len(episode_profits))),
    "Profit": episode_profits,
    "Reward": episode_rewards
})

df_metrics.to_csv("training_metrics.csv", index=False)
print("Training metrics saved to CSV")

